# UinFit-S 로고 만들기 (GPU)
위에서부터 **셀을 하나씩** 선택하고 `Shift + Enter`를 눌러 실행하세요.

## 1단계. GPU가 켜졌는지 확인

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout or 'GPU를 찾지 못했어요')
try:
    import torch
    print('torch GPU 사용 가능:',torch.cuda.is_available())
    if torch.cuda.is_available(): print('GPU 이름:',torch.cuda.get_device_name(0))
except ImportError:
    print('torch가 없어요 → 2단계에서 설치해요')

## 2단계. 필요한 도구 설치 (이미 있으면 금방 끝나요)

In [ ]:
!pip install -q diffusers transformers accelerate safetensors pillow

## 3단계. 그림 그리는 AI 불러오기
처음에는 모델을 내려받느라 몇 분 걸려요. 인터넷이 막혀 있으면 오류가 나요. 그때는 대회 안내에 있는 모델 위치를 알려 주세요.

In [ ]:
import torch
from diffusers import AutoPipelineForText2Image
pipe=AutoPipelineForText2Image.from_pretrained('stabilityai/sdxl-turbo',torch_dtype=torch.float16,variant='fp16').to('cuda')
print('준비 완료')

## 4단계. 로고 배경·심볼 그림 만들기
AI는 글씨를 자주 틀리게 그려요. 그래서 **그림만** 만들고 'UinFit-S' 글씨는 5단계에서 직접 넣어요.

In [ ]:
import os
os.makedirs('uinfit_images',exist_ok=True)
prompts={
 'logo_symbol':'minimal flat vector logo symbol, graduation cap combined with a checkmark and a spark, blue and navy colors, white background, no text, clean simple shape',
 'banner_bg':'soft lavender and blue abstract gradient background, subtle academic icons, wide banner, minimal, no text',
 'card_scholarship':'flat illustration of a scholarship certificate and coins, blue tones, no text',
 'card_activity':'flat illustration of students teamwork and a megaphone, pink tones, no text',
 'card_contest':'flat illustration of a trophy and a lightbulb, green tones, no text',
}
for name,p in prompts.items():
    img=pipe(prompt=p,num_inference_steps=4,guidance_scale=0.0,height=512,width=512).images[0]
    img.save(f'uinfit_images/{name}.png'); print('저장:',name)

## 5단계. 'UinFit-S' 글씨 넣어서 최종 로고 만들기

In [ ]:
from PIL import Image,ImageDraw,ImageFont
import glob
sym=Image.open('uinfit_images/logo_symbol.png').convert('RGB').resize((256,256))
canvas=Image.new('RGB',(900,300),'white')
canvas.paste(sym,(20,22))
fonts=glob.glob('/usr/share/fonts/**/*Bold*.ttf',recursive=True)+glob.glob('/usr/share/fonts/**/*.ttf',recursive=True)
font=ImageFont.truetype(fonts[0],110) if fonts else ImageFont.load_default()
ImageDraw.Draw(canvas).text((300,85),'UinFit-S',fill=(47,91,255),font=font)
canvas.save('uinfit_images/logo_final.png'); canvas

## 6단계. 한 번에 보기 + 압축하기
만든 그림이 마음에 안 들면 4단계의 영어 설명을 바꿔서 다시 실행하세요.

In [ ]:
import shutil,glob
from IPython.display import display
for f in sorted(glob.glob('uinfit_images/*.png')):
    print(f); display(Image.open(f).resize((256,256)))
shutil.make_archive('uinfit_images','zip','uinfit_images')
print('uinfit_images.zip 이 왼쪽 파일 목록에 생겼어요')

## 7단계. 이메일로 보내기
1. 왼쪽 파일 목록에서 `uinfit_images.zip`을 우클릭 → **Download**
2. CUBEApp의 **크롬**을 열고 메일에 첨부해서 보내요.
3. 받은 이미지를 프로젝트의 `images/` 폴더에 넣어요.